# Projection as Knowledge Re-Representation

**Course:** MAI554 – Deep Learning for NLP | **Lecture:** 05 | **Spring 2026** | **Prof. Anis Koubaa**

---

### Learning Objectives

1. **Understand projection as a coordinate transform** — the same knowledge, re-expressed so that different aspects become dominant
2. **See the PCA analogy** — eigenvectors find data-driven axes; Transformers learn task-driven axes
3. **Implement Q/K/V projections** and visualise how the same embedding becomes three specialised views
4. **Understand why projection matters** — feature extraction, demand-supply matching, and layered abstraction
5. **See projection everywhere** — Q/K/V, FFN, output layer are all the same operation: $y = xW$

> **Central idea:** A linear projection $W \cdot x$ does **not create or destroy** information — it **re-expresses** it in a coordinate system where the right features are on the surface. Every matrix multiplication in the Transformer is a projection.

## Setup

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt, math
from matplotlib.patches import FancyArrowPatch
plt.rcParams['figure.dpi'] = 100
torch.manual_seed(42); np.random.seed(42)

---

## Part 1: Projection = Change of Coordinate System

### The GPS vs Robot Analogy

Imagine a robot standing in Riyadh. A person is 3 metres ahead. Where is that person?

- **Ask a cartographer (GPS):** $(24.71°N, 46.68°E)$ — universal, precise, but the robot can't easily compute "how far to reach?"
- **Ask the robot (local frame):** $(3m, 1m)$ — task-specific, the answer is immediate

**Both answers describe the same person.** The difference is the **coordinate system**. Going from GPS to robot-local is a **projection** — a matrix multiplication $\vec{p}_{robot} = T \cdot \vec{p}_{GPS}$ that rotates and translates the axes.

**The Transformer parallel:**
- **Raw embedding** = GPS coordinates (knows everything about the word, but nothing is emphasised)
- **$W_Q \cdot x$** = project into Query space ("what do I need?" becomes readable)
- **$W_K \cdot x$** = project into Key space ("what can I offer?" becomes readable)
- **$W_V \cdot x$** = project into Value space ("what content do I carry?" becomes readable)

Same data. Same knowledge. Just re-expressed so the **right features are on the surface**.

**Seeing it with code.** We create a 2D point in "GPS" coordinates and apply a rotation matrix to move it into "robot-local" coordinates. The point doesn't move — only the axes change.

In [ ]:
# --- Projection = rotation: same point, different coordinate system ---
theta = math.radians(35)  # robot is rotated 35° relative to GPS
R = torch.tensor([[math.cos(theta), math.sin(theta)],
                  [-math.sin(theta), math.cos(theta)]])  # rotation matrix

p_gps = torch.tensor([3.0, 4.0])    # point in GPS coordinates
p_robot = R @ p_gps                  # same point in robot coordinates

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: GPS frame
ax1.set_title('GPS Frame (universal)', fontsize=12)
ax1.arrow(0, 0, 5, 0, head_width=0.15, color='gray', alpha=0.3)
ax1.arrow(0, 0, 0, 5, head_width=0.15, color='gray', alpha=0.3)
ax1.plot(*p_gps, 'ro', ms=12, zorder=5)
ax1.annotate(f'({p_gps[0]:.1f}, {p_gps[1]:.1f})', p_gps.numpy()+0.2, fontsize=11, color='red')
ax1.set_xlim(-1, 6); ax1.set_ylim(-1, 6); ax1.set_aspect('equal')
ax1.set_xlabel('East'); ax1.set_ylabel('North')

# Right: Robot frame (rotated axes)
ax2.set_title('Robot Frame (task-specific)', fontsize=12)
ax2.arrow(0, 0, 5*math.cos(theta), 5*math.sin(theta), head_width=0.15, color='green', alpha=0.4)
ax2.arrow(0, 0, -5*math.sin(theta), 5*math.cos(theta), head_width=0.15, color='green', alpha=0.4)
ax2.plot(*p_gps, 'ro', ms=12, zorder=5)
ax2.annotate(f'Robot sees: ({p_robot[0]:.1f}, {p_robot[1]:.1f})', p_gps.numpy()+0.2, fontsize=11, color='green')
ax2.set_xlim(-1, 6); ax2.set_ylim(-1, 6); ax2.set_aspect('equal')
ax2.set_xlabel('Robot Forward'); ax2.set_ylabel('Robot Left')

plt.suptitle('Same point, different coordinate system — that is projection', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()
print(f"GPS:   ({p_gps[0]:.1f}, {p_gps[1]:.1f})")
print(f"Robot: ({p_robot[0]:.1f}, {p_robot[1]:.1f})  ← same point, easier to use for the robot's task")

---

## Part 2: The PCA Analogy — Finding the "Natural" Axes of Data

PCA (Principal Component Analysis) is the simplest form of projection. It asks: **what coordinate system makes this data easiest to understand?**

1. **Compute the covariance matrix** $C = \frac{1}{n} X^T X$ — this tells us which features co-vary
2. **Find the eigenvectors** $C\vec{v} = \lambda\vec{v}$ — these are the directions where data has the most spread
3. **Project** $z = W^T x$ where $W = [\vec{v}_1, \vec{v}_2, \ldots]$ — rotate the data so these directions become the axes

The result: features that were **correlated and tangled** become **independent and sorted by importance**.

**PCA visualised.** We generate correlated 2D data, find the eigenvectors (principal components), and project onto them. Left: the data in its original tangled form. Right: the same data after projection — now aligned with the axes, with PC1 capturing the most variance.

In [ ]:
# --- PCA: rotate data to its natural axes ---
# Generate correlated 2D data
cov = np.array([[2.0, 1.5], [1.5, 1.0]])
data = np.random.multivariate_normal([0, 0], cov, 200)

# Step 1: Covariance matrix
C = np.cov(data.T)
# Step 2: Eigenvectors and eigenvalues
eigenvalues, eigenvectors = np.linalg.eigh(C)
idx = eigenvalues.argsort()[::-1]  # sort by largest eigenvalue
eigenvalues, eigenvectors = eigenvalues[idx], eigenvectors[:, idx]
# Step 3: Project
projected = data @ eigenvectors

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: original space with eigenvectors shown
ax1.scatter(data[:, 0], data[:, 1], alpha=0.4, s=20, c='steelblue')
for i, (ev, lam, c) in enumerate(zip(eigenvectors.T, eigenvalues, ['#e74c3c', '#2ecc71'])):
    ax1.arrow(0, 0, ev[0]*lam*1.5, ev[1]*lam*1.5, head_width=0.1, color=c, lw=2.5)
    ax1.text(ev[0]*lam*1.7, ev[1]*lam*1.7, f'v{i+1} (λ={lam:.2f})', color=c, fontsize=11, fontweight='bold')
ax1.set_title('Original Space\n(features correlated, tangled)', fontsize=12)
ax1.set_xlabel('x₁'); ax1.set_ylabel('x₂'); ax1.set_aspect('equal')
ax1.set_xlim(-5, 5); ax1.set_ylim(-4, 4)

# Right: projected space
ax2.scatter(projected[:, 0], projected[:, 1], alpha=0.4, s=20, c='steelblue')
ax2.set_title('After Projection (z = Wᵀx)\n(axes aligned with data structure)', fontsize=12)
ax2.set_xlabel('PC₁ (most variance)'); ax2.set_ylabel('PC₂ (least variance)')
ax2.set_aspect('equal'); ax2.set_xlim(-5, 5); ax2.set_ylim(-4, 4)

plt.suptitle('PCA = rotating to the coordinate system where the data is simplest', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

print(f"Eigenvalues: λ₁={eigenvalues[0]:.2f} (big spread), λ₂={eigenvalues[1]:.2f} (small spread)")
print("After projection, the correlation is gone — each axis captures independent information.")

---

## Part 3: From PCA to Transformers — Fixed vs Learned Axes

PCA and Transformers both perform the **same operation**: multiply by a matrix $W$ to rotate into a better coordinate system. The difference:

| | PCA | Transformer |
|---|---|---|
| **Matrix** | $W = [\vec{v}_1, \ldots, \vec{v}_k]$ (eigenvectors of covariance) | $W_Q, W_K, W_V$ (learned by gradient descent) |
| **How found** | Data statistics (unsupervised) | Task supervision (labels guide learning) |
| **How many** | One fixed projection for all purposes | Three different projections per token |
| **Goal** | Best general representation (max variance) | Best Q/K/V roles (seek / offer / share) |

> **Shared principle:** Multiply by a matrix $W$ = rotate into a space where the right features are exposed. PCA: eigenvectors expose variance. Transformers: $W_Q/W_K/W_V$ expose query/key/value roles.

---

## Part 4: Q/K/V — One Embedding, Three Spaces

The embedding of "animal" contains **everything** the model knows: syntax (noun, singular), semantics (living, moves), relations (can be subject, referent), properties (can be tired). But all of this knowledge is **tangled together** in one vector.

Three learned projection matrices $W_Q, W_K, W_V$ each select a **different subset** of this knowledge:

- **Query space** ($W_Q$): "What do I **need**?" — e.g. I need a verb, I need a determiner
- **Key space** ($W_K$): "What can I **offer**?" — e.g. I am a noun, I am a referent for pronouns
- **Value space** ($W_V$): "What content do I **carry**?" — e.g. living-thing features, semantic role = agent

No information is created — it is **reorganised** into specialised coordinate systems where different semantic aspects become the **dominant axes**.

**Three projections from the same source.** We take a single embedding vector $x$ and project it through three different learned matrices. Each output emphasises different features — visible as different heatmap patterns from the same input.

In [ ]:
# --- One embedding → three different projections ---
d_model, d_k = 16, 8
sentence = ["The", "animal", "didn't", "cross", "the", "street"]

# Simulate token embeddings (in practice these come from nn.Embedding)
torch.manual_seed(42)
X = torch.randn(len(sentence), d_model)  # (6, 16) — embedding for each token

# Three learned projection matrices
W_Q = nn.Linear(d_model, d_k, bias=False)
W_K = nn.Linear(d_model, d_k, bias=False)
W_V = nn.Linear(d_model, d_k, bias=False)

Q = W_Q(X)  # (6, 8) — Query space: "what do I need?"
K = W_K(X)  # (6, 8) — Key space:   "what can I offer?"
V = W_V(X)  # (6, 8) — Value space: "what do I carry?"

# Visualise: same input, three different outputs
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, data, title, color in zip(axes,
    [X.detach(), Q.detach(), K.detach(), V.detach()],
    ['Embedding X\n(general knowledge)', 'Q = X·W_Q\n("What do I need?")',
     'K = X·W_K\n("What can I offer?")', 'V = X·W_V\n("What do I carry?")'  ],
    ['Blues', 'Purples', 'Greens', 'Oranges']):
    im = ax.imshow(data.numpy(), cmap=color, aspect='auto')
    ax.set_yticks(range(len(sentence))); ax.set_yticklabels(sentence)
    ax.set_xlabel('Dimension'); ax.set_title(title, fontsize=10)
    plt.colorbar(im, ax=ax)
plt.suptitle('Same knowledge, three different spaces', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

print(f"X shape: {X.shape} → Q: {Q.shape}, K: {K.shape}, V: {V.shape}")
print("Each W selects different features — like viewing the same scene through different lenses.")

---

## Part 5: Projection Changes What "Nearby" Means

In the raw embedding space, "it" and "animal" may be **far apart** — they are different words with different surface forms. But a pronoun **needs** its referent, and a referent **offers** itself to pronouns.

After projection into Query and Key spaces, "it" and "animal" become **neighbours** — because the projection reveals that they are complementary: one needs what the other offers.

**Similarity before and after projection.** We compute cosine similarity between all token pairs in the raw embedding space and in the projected Q/K spaces. Notice how the similarity structure changes — projection reorganises what the model considers "close".

In [ ]:
# --- Projection changes what "nearby" means ---
def cosine_sim(M):
    """Cosine similarity matrix for rows of M."""
    M_norm = M / M.norm(dim=1, keepdim=True)
    return (M_norm @ M_norm.T).detach().numpy()

sim_X = cosine_sim(X)       # similarity in raw embedding space
sim_Q = cosine_sim(Q)       # similarity in Query space
sim_K = cosine_sim(K)       # similarity in Key space

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, sim, title in zip([ax1, ax2, ax3], [sim_X, sim_Q, sim_K],
    ['Raw Embedding Space', 'Query Space (W_Q)', 'Key Space (W_K)']):
    im = ax.imshow(sim, cmap='RdBu_r', vmin=-1, vmax=1, aspect='equal')
    ax.set_xticks(range(len(sentence))); ax.set_xticklabels(sentence, rotation=45, ha='right')
    ax.set_yticks(range(len(sentence))); ax.set_yticklabels(sentence)
    ax.set_title(title, fontsize=11); plt.colorbar(im, ax=ax)
plt.suptitle('Projection changes which tokens are "close" to each other', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

print("Raw space: similarity depends on surface form (word identity).")
print("After projection: similarity depends on ROLE — what each token needs or offers.")

---

## Part 6: Demand-Supply Matching via Q/K Projection

Q/K projections enable a **demand-supply** mechanism:
- **Q** ("what do I need?"): "it" needs a referent
- **K** ("what can I offer?"): "animal" offers itself as a referent

The attention score $q_i \cdot k_j$ measures **match quality**: high dot product means demand meets supply. Then the **Value payload** from "animal" is transferred to "it" — enriching "it" with knowledge it could never access alone.

This is why projection is not just a technicality — **without it, the model cannot distinguish what a token needs from what it offers**.

**Attention scores = Q·K matching.** We compute $QK^T / \sqrt{d_k}$ and apply softmax to see which tokens each query attends to. High attention weight means Q's demand matches K's supply.

In [ ]:
# --- Q·K attention = demand-supply matching ---
scores = (Q @ K.T) / math.sqrt(d_k)                # (6, 6) raw scores
attn_weights = torch.softmax(scores, dim=-1)         # normalise to probabilities
output = attn_weights @ V                            # weighted sum of values

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: attention weights
im1 = ax1.imshow(attn_weights.detach().numpy(), cmap='Blues', aspect='equal', vmin=0, vmax=0.5)
ax1.set_xticks(range(len(sentence))); ax1.set_xticklabels(sentence, rotation=45, ha='right')
ax1.set_yticks(range(len(sentence))); ax1.set_yticklabels(sentence)
ax1.set_title('Attention Weights (softmax of QKᵀ/√d)\nEach row: who does this token attend to?')
ax1.set_xlabel('Key (offers)'); ax1.set_ylabel('Query (needs)')
for i in range(len(sentence)):
    for j in range(len(sentence)):
        ax1.text(j, i, f'{attn_weights[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im1, ax=ax1)

# Right: output = enriched representation
im2 = ax2.imshow(output.detach().numpy(), cmap='Oranges', aspect='auto')
ax2.set_yticks(range(len(sentence))); ax2.set_yticklabels(sentence)
ax2.set_xlabel('Dimension'); ax2.set_title('Output = Attention × V\n(each token enriched with context)')
plt.colorbar(im2, ax=ax2)

plt.tight_layout(); plt.show()
print("Each row of the attention matrix shows WHERE a token looks for information.")
print("The output is a weighted blend of Value vectors — contextual knowledge enrichment.")

---

## Part 7: The FFN as Projection — Expand, Transform, Compress

The Feed-Forward Network (FFN) after attention is **two projections back-to-back**:

$$\text{FFN}(x) = \text{ReLU}(x W_1 + b_1) W_2 + b_2$$

1. **$W_1$ projects UP** to a wider space ($d_{model} \to 4 \times d_{model}$) — like zooming in to see fine details
2. **ReLU** kills negative dimensions — creating sparse, non-linear feature combinations
3. **$W_2$ projects BACK DOWN** to the original size — keeping only the useful combinations

Each hidden neuron learns to detect one "knowledge pattern" — e.g. "is this a country?" or "is this past tense?" The expand-then-compress structure gives the model room to **combine features non-linearly** before projecting back to the residual stream.

**FFN expand-compress visualised.** We pass a single token through an FFN and show the dimensionality at each stage. Notice the wide hidden layer (4x expansion) and the sparse activation after ReLU (many neurons are zero).

In [ ]:
# --- FFN: expand → ReLU → compress ---
d_model, d_ff = 32, 128  # typical: d_ff = 4 * d_model

W1 = nn.Linear(d_model, d_ff)
W2 = nn.Linear(d_ff, d_model)

x = torch.randn(1, d_model)          # single token
hidden = torch.relu(W1(x))           # expand + ReLU
out = W2(hidden)                      # compress back

# Visualise the three stages
fig, axes = plt.subplots(3, 1, figsize=(14, 5))

axes[0].barh(range(d_model), x.detach().squeeze().numpy(), color='steelblue', height=0.8)
axes[0].set_title(f'Input: d_model = {d_model}', fontsize=10); axes[0].set_xlim(-3, 3)

axes[1].barh(range(d_ff), hidden.detach().squeeze().numpy(), color='#2ecc71', height=0.8)
axes[1].set_title(f'Hidden after ReLU: d_ff = {d_ff} (4x expansion, {(hidden.squeeze() == 0).sum().item()}/{d_ff} neurons dead)', fontsize=10)
axes[1].set_xlim(-3, 3)

axes[2].barh(range(d_model), out.detach().squeeze().numpy(), color='#e67e22', height=0.8)
axes[2].set_title(f'Output: d_model = {d_model} (compressed back)', fontsize=10); axes[2].set_xlim(-3, 3)

for ax in axes: ax.set_ylabel('Dim')
plt.tight_layout(); plt.show()

pct_dead = (hidden.squeeze() == 0).sum().item() / d_ff * 100
print(f"ReLU killed {pct_dead:.0f}% of hidden neurons → sparse activation.")
print("The FFN is two projections: W₁ expands to find features, W₂ compresses to keep only what matters.")

---

## Part 8: Projection Is Everywhere in the Transformer

Every matrix multiplication in the Transformer is a projection — the **same operation** ($y = xW$), used for **different purposes**:

| Use | Formula | Purpose |
|---|---|---|
| **Q/K/V Projection** | $Q = XW_Q$, $K = XW_K$, $V = XW_V$ | Re-represent each token into three roles: seek, offer, share |
| **FFN** | $\text{ReLU}(xW_1)W_2$ | Project up to find non-linear feature combinations, project back |
| **Output Layer** | $\text{logits} = hW_{vocab}$ | Project from hidden space ($d_{model}$) to vocabulary (30k-100k words) |
| **PCA (analogy)** | $z = W_{eigvec}^T x$ | Project onto eigenvectors — same principle, but data-driven not task-driven |

> **Every projection is the same operation: multiply by a learned matrix.** Q/K/V re-represent for attention; FFN re-represents for reasoning; the final layer re-represents for prediction.

---

## Summary & Key Takeaways

| Concept | Key Point |
|---|---|
| **Projection = coordinate transform** | $Wx$ rotates into a space where the right features are on the surface |
| **Nothing is created or destroyed** | Same knowledge, re-expressed so that invisible aspects become dominant |
| **PCA → Transformer** | PCA finds fixed data-driven axes; Transformers learn task-driven axes |
| **Q/K/V = three lenses** | Same embedding → Query (what I need), Key (what I offer), Value (what I carry) |
| **Projection changes "nearby"** | Raw: "it" and "animal" are far; Projected: they become neighbours |
| **Demand-supply matching** | Q·K score = how well demand meets supply; V transfers the payload |
| **FFN = expand-compress** | $W_1$ projects up to find features, ReLU makes them sparse, $W_2$ projects back |
| **Everywhere** | Q/K/V, FFN, output layer — all are $y = xW$, just different learned $W$ |

> **The big picture:** Embedding → Projection (re-represent into Q/K/V) → Attention (match & blend) → Enriched knowledge. The output knows more than any single input.